# Train the traffic-sign **detector** (Stage 1)

This notebook trains a small YOLOv8 model that answers one question: **where are the traffic signs in this image?**
It is trained with a single class, `sign` — naming the sign is still done by your existing GTSRB classifier (`traffic_sign_model.h5`).

**Run it on Google Colab with a GPU:** `Runtime → Change runtime type → T4 GPU`. Training takes about 30–60 minutes.

At the end you download **`detector.pt`** and put it in the project root (next to `traffic_sign_model.h5`). The backend picks it up automatically.

In [ ]:
!pip install -q ultralytics
import ultralytics
ultralytics.checks()

## 1. Download GTSDB (German Traffic Sign Detection Benchmark)
900 road photos (1360×800) with bounding boxes for every sign, ~1.6 GB.

In [ ]:
!wget -q --show-progress https://sid.erda.dk/public/archives/ff17dc924eba88d5d01a807357d6614c/FullIJCNN2013.zip
!unzip -q -o FullIJCNN2013.zip
!head -5 FullIJCNN2013/gt.txt

## 2. Convert to YOLO format
`gt.txt` lines look like `00000.ppm;x1;y1;x2;y2;classId`. YOLO wants one `.txt` per image with `class cx cy w h` (normalised 0–1).

* Every box becomes class `0` (`sign`) — the detector only needs to *find* signs.
* Images with no signs are kept with an empty label file. These "background" images teach the model what is **not** a sign (fewer false boxes on hands, faces, walls…).
* Official split: images `00000–00599` train, `00600–00899` validation.

In [ ]:
import os, cv2
from collections import defaultdict

SRC = "FullIJCNN2013"
OUT = "gtsdb_yolo"

boxes = defaultdict(list)
with open(f"{SRC}/gt.txt") as f:
    for line in f:
        name, x1, y1, x2, y2, _cls = line.strip().split(";")
        boxes[name].append(tuple(map(int, (x1, y1, x2, y2))))

n_boxes = 0
for i in range(900):
    name = f"{i:05d}.ppm"
    split = "train" if i < 600 else "val"
    os.makedirs(f"{OUT}/images/{split}", exist_ok=True)
    os.makedirs(f"{OUT}/labels/{split}", exist_ok=True)

    img = cv2.imread(f"{SRC}/{name}")          # YOLO doesn't read .ppm → save as .jpg
    h, w = img.shape[:2]
    cv2.imwrite(f"{OUT}/images/{split}/{i:05d}.jpg", img, [cv2.IMWRITE_JPEG_QUALITY, 95])

    with open(f"{OUT}/labels/{split}/{i:05d}.txt", "w") as lf:
        for x1, y1, x2, y2 in boxes.get(name, []):
            cx, cy = (x1 + x2) / 2 / w, (y1 + y2) / 2 / h
            bw, bh = (x2 - x1) / w, (y2 - y1) / h
            lf.write(f"0 {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}\n")
            n_boxes += 1

print(f"Converted 900 images, {n_boxes} boxes, {900 - len(boxes)} background images")

with open("signs.yaml", "w") as f:
    f.write(f"path: {os.path.abspath(OUT)}\ntrain: images/train\nval: images/val\nnames:\n  0: sign\n")
print(open("signs.yaml").read())

## 3. (Optional, recommended) Add your own webcam images

GTSDB is photos taken from a car. If you demo by **holding a printed sign / phone in front of a webcam**, the detector has never seen that — hands, fingers, a room behind the sign, screen glare.
Adding even **100–200 of your own webcam frames** makes a big difference:

1. Take screenshots from your webcam: signs held near and far, partly covered by fingers, different lighting, plus ~30 frames with **no sign** (just you/your hands/the room).
2. Label them at [roboflow.com](https://roboflow.com) (free) — draw a box around every sign, use one class called `sign`. Leave no-sign images unlabelled.
3. Export as **YOLOv8** format, upload the zip here, and set `CUSTOM_ZIP` below.

Skip this cell if you don't have custom data yet — you can come back and retrain later.

In [ ]:
import glob, shutil, zipfile

CUSTOM_ZIP = ""   # e.g. "/content/my_webcam_signs.zip"  (Roboflow YOLOv8 export)

if CUSTOM_ZIP:
    with zipfile.ZipFile(CUSTOM_ZIP) as z:
        z.extractall("custom")
    added = 0
    for split_in, split_out in [("train", "train"), ("valid", "val"), ("test", "val")]:
        for img in glob.glob(f"custom/{split_in}/images/*"):
            stem = os.path.splitext(os.path.basename(img))[0]
            shutil.copy(img, f"{OUT}/images/{split_out}/custom_{os.path.basename(img)}")
            src_lbl = f"custom/{split_in}/labels/{stem}.txt"
            dst_lbl = f"{OUT}/labels/{split_out}/custom_{stem}.txt"
            lines = open(src_lbl).read().split("\n") if os.path.exists(src_lbl) else []
            with open(dst_lbl, "w") as f:   # force every class to 0 ("sign")
                f.writelines("0 " + " ".join(l.split()[1:]) + "\n" for l in lines if l.strip())
            added += 1
    print(f"Added {added} custom images")
else:
    print("No custom data – training on GTSDB only")

## 4. Train

* `yolov8n` = the smallest/fastest model — runs in real time on a laptop CPU.
* `imgsz=1024`: GTSDB signs are small (16–128 px). Training at a higher resolution keeps them visible, which is also what helps with **far-away signs**.
* Default augmentations (mosaic, scale, colour jitter) simulate different distances and lighting. Horizontal flips are fine here because the detector doesn't care *which* sign it is — left/right arrows are told apart by the classifier, which never sees flipped images.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")   # start from COCO-pretrained weights
results = model.train(
    data="signs.yaml",
    epochs=80,
    imgsz=1024,
    batch=16,
    patience=20,
    scale=0.6,        # stronger zoom in/out → more robust to distance
    degrees=5,        # slight rotation (tilted sign held in hand)
    name="sign_detector",
)

## 5. Check results

In [ ]:
best = YOLO("runs/detect/sign_detector/weights/best.pt")
metrics = best.val(data="signs.yaml", imgsz=1024)
print(f"mAP50: {metrics.box.map50:.3f}   mAP50-95: {metrics.box.map:.3f}")

In [ ]:
# Show a few validation images with predicted boxes
from IPython.display import Image, display
preds = best.predict(sorted(glob.glob(f"{OUT}/images/val/*.jpg"))[:6], conf=0.35, imgsz=1024, save=True)
for p in sorted(glob.glob(f"{preds[0].save_dir}/*.jpg"))[:6]:
    display(Image(p, width=700))

## 6. Download `detector.pt`
On **Kaggle** click the link printed below (or find it under *Output → /kaggle/working* in the right sidebar). On **Colab** the download starts automatically.

Put it in the project root (same folder as `traffic_sign_model.h5`), `pip install ultralytics` in the backend environment, and restart the backend. `GET /health` should show `"detector_loaded": true`.

In [ ]:
import os, shutil
shutil.copy("runs/detect/sign_detector/weights/best.pt", "detector.pt")

if os.path.exists("/kaggle/working"):
    # Kaggle: files in /kaggle/working appear in the Output panel (right sidebar).
    # The link below also lets you download it directly.
    shutil.copy("detector.pt", "/kaggle/working/detector.pt")
    from IPython.display import FileLink, display
    os.chdir("/kaggle/working")
    display(FileLink("detector.pt"))
else:
    from google.colab import files
    files.download("detector.pt")